# Forest & GDP Database — Build Documentation

**Project:** World Forest Change & GDP Correlation Analysis  
**Database:** db_forestgdp (MSSQL Server)
**Team:** [Kucerova Kristina], [Kmetova Barbara]
**Date:** May 2026

## Overview
This notebook documents the full process of building the database schema,
dimension tables, and fact tables used for the analysis. Raw data was sourced
from Kaggle (FAO, World Bank, Global Forest Watch).

## Schema structure
- `raw` — source tables loaded as-is from Kaggle
- `dbo` — cleaned dimension and fact tables

In [2]:
--Create the country dimension table
CREATE TABLE dbo.dim_country (
    country_code    VARCHAR(10)     NOT NULL PRIMARY KEY,  --ISO3 code
    country_name    NVARCHAR(100),                         -- Full country name
    region          NVARCHAR(100),                         -- World Bank region
    income_group    NVARCHAR(50)                           -- World Bank income classification
);

--Populate from Forest_year joined with income groups
-- EFT JOIN ensures countries without WB classification are still included
INSERT INTO dbo.dim_country (country_code, country_name, region, income_group)
SELECT DISTINCT
    f.Code              AS country_code,
    f.Country           AS country_name,
    i.region,
    i.income_group
FROM raw.Forest_year f
LEFT JOIN raw.stg_gdp_income i ON i.country_code = f.Code
WHERE f.Code IS NOT NULL;  --Exclude rows with no country code

Msg 2627, Level 14, State 1, Line 11
Violation of PRIMARY KEY constraint 'PK__dim_coun__3436E9A4649D5BF5'. Cannot insert duplicate key in object 'dbo.dim_country'. The duplicate key value is ().

The statement has been terminated.

Total execution time: 00:00:00.101